In [33]:
import pandas as pd

In [2]:
df = pd.read_csv("Synthetic_Financial_datasets_log.csv")

In [9]:
df = df.drop(columns = "isFlaggedFraud")

#### 1. Data Type Validation

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 706.2 MB


#### 2. Missing Value check

In [34]:
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    "Missing": missing_count,
    "Missing (%)": missing_percent
})

print(missing_summary)

                Missing  Missing (%)
step                  0          0.0
type                  0          0.0
amount                0          0.0
nameOrig              0          0.0
oldbalanceOrg         0          0.0
newbalanceOrig        0          0.0
nameDest              0          0.0
oldbalanceDest        0          0.0
newbalanceDest        0          0.0
isFraud               0          0.0


#### 3. Remove Duplicates

In [20]:
df.duplicated().sum()

np.int64(0)

#### 4. Validate Invalid Values

In [35]:
df.query("oldbalanceDest <0 or newbalanceDest < 0 or newbalanceOrig < 0 or oldbalanceOrg <0").count()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
dtype: int64

#### 5. Analyze Outliers

In [38]:
import pandas as pd

num_cols = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

outlier_summary = []

for col in num_cols:
    q25 = df[col].quantile(0.25)
    q50 = df[col].quantile(0.50)
    q75 = df[col].quantile(0.75)
    iqr = q75 - q25

    lower = q25 - 1.5 * iqr
    upper = q75 + 1.5 * iqr

    outlier_count = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_summary.append({
        "Column": col,
        "25%": q25,
        "50%": q50,
        "75%": q75,
        "IQR": iqr,
        "Lower Bound": lower,
        "Upper Bound": upper,
        "Outlier Count": outlier_count,
        "Outlier (%)": round(outlier_count / len(df) * 100, 2)
    })

outlier_summary = pd.DataFrame(outlier_summary)

outlier_summary

,Column,25%,50%,75%,IQR,Lower Bound,Upper Bound,Outlier Count,Outlier (%)
0,step,156.00,239.000,3.350000e+02,1.790000e+02,-1.125000e+02,6.035000e+02,102688,1.61
1,amount,13389.57,74871.940,2.087215e+05,1.953319e+05,-2.796083e+05,5.017193e+05,338078,5.31
2,oldbalanceOrg,0.00,14208.000,1.073152e+05,1.073152e+05,-1.609728e+05,2.682879e+05,1112507,17.49
3,newbalanceOrig,0.00,0.000,1.442584e+05,1.442584e+05,-2.163876e+05,3.606460e+05,1053391,16.56
4,oldbalanceDest,0.00,132705.665,9.430367e+05,9.430367e+05,-1.414555e+06,2.357592e+06,786135,12.36
5,newbalanceDest,0.00,214661.440,1.111909e+06,1.111909e+06,-1.667864e+06,2.779773e+06,738527,11.61


#### 6. Check Consistency

In [41]:
# Check unique values of transaction type
print(df["type"].unique())

# Check target values
print(df["isFraud"].unique())


<ArrowStringArray>
['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN']
Length: 5, dtype: str
[0 1]


#### 7. Validate Business Logic

In [42]:
# Sender balance should not be negative
invalid_origin_balance = df[
    (df["oldbalanceOrg"] < 0) |
    (df["newbalanceOrig"] < 0)
]

# Receiver balance should not be negative
invalid_dest_balance = df[
    (df["oldbalanceDest"] < 0) |
    (df["newbalanceDest"] < 0)
]

print("Invalid origin balance:", len(invalid_origin_balance))
print("Invalid destination balance:", len(invalid_dest_balance))

Invalid origin balance: 0
Invalid destination balance: 0


In [12]:
df["type"].drop_duplicates()

0       PAYMENT
2      TRANSFER
3      CASH_OUT
9         DEBIT
389     CASH_IN
Name: type, dtype: str

In [18]:
df[df["isFraud"]==1].count()

step              8213
type              8213
amount            8213
nameOrig          8213
oldbalanceOrg     8213
newbalanceOrig    8213
nameDest          8213
oldbalanceDest    8213
newbalanceDest    8213
isFraud           8213
dtype: int64

In [19]:
df

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0
...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1


In [32]:
df.query("isFraud == 1")

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
2,1,TRANSFER,181.00,C1305486145,181.00,0.0,C553264065,0.00,0.00,1
3,1,CASH_OUT,181.00,C840083671,181.00,0.0,C38997010,21182.00,0.00,1
251,1,TRANSFER,2806.00,C1420196421,2806.00,0.0,C972765878,0.00,0.00,1
252,1,CASH_OUT,2806.00,C2101527076,2806.00,0.0,C1007251739,26202.00,0.00,1
680,1,TRANSFER,20128.00,C137533655,20128.00,0.0,C1848415041,0.00,0.00,1
...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.0,C776919290,0.00,339682.13,1
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.0,C1881841831,0.00,0.00,1
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.0,C1365125890,68488.84,6379898.11,1
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.0,C2080388513,0.00,0.00,1
